In [1]:
import sys, io, contextlib, re, csv
import torch, torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
import fedml
from fedml import FedMLRunner
print("Импорты готовы")

E:\proekt3\practika_leto\venv-fedml\Lib\site-packages\requests\__init__.py:109: RequestsDependencyWarning: urllib3 (2.7.0) or chardet (7.4.3)/charset_normalizer (3.4.9) doesn't match a supported version!
  warnings.warn(
E:\proekt3\practika_leto\venv-fedml\Lib\site-packages\fedml\computing\scheduler\comm_utils\sys_utils.py:17: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


Импорты готовы


In [2]:
config = """common_args:
  training_type: "simulation"
  random_seed: 0
data_args:
  dataset: "mnist"
  data_cache_dir: "./data"
  partition_method: "homo"
model_args:
  model: "lr"
train_args:
  federated_optimizer: "FedAvg"
  client_id_list:
  client_num_in_total: 3
  client_num_per_round: 3
  comm_round: 5
  epochs: 1
  batch_size: 64
  client_optimizer: sgd
  learning_rate: 0.03
  weight_decay: 0.001
validation_args:
  frequency_of_the_test: 1
device_args:
  using_gpu: false
  gpu_id: 0
comm_args:
  backend: "sp"
tracking_args:
  enable_wandb: false
  log_file_dir: "./log"
"""
with open("fedml_config_3clients.yaml", "w", encoding="utf-8") as f:
    f.write(config)
print("Конфиг записан")

Конфиг записан


In [3]:
import logging

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(784, 128)
        self.fc2 = nn.Linear(128, 10)
    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.view(x.size(0), -1))))

def to_batches(data, batch_size, shuffle):
    loader = DataLoader(data, batch_size=batch_size, shuffle=shuffle)
    return [(x, y) for x, y in loader]

def build_3client_mnist(batch_size):
    tfm = transforms.Compose([transforms.ToTensor(),
                              transforms.Normalize((0.1307,), (0.3081,))])
    train_set = datasets.MNIST("./data", train=True,  download=True, transform=tfm)
    test_set  = datasets.MNIST("./data", train=False, download=True, transform=tfm)
    g = torch.Generator().manual_seed(0)
    idx = torch.randperm(len(train_set), generator=g).tolist()
    per = len(idx) // 3
    shards = [idx[0:per], idx[per:2*per], idx[2*per:3*per]]
    test_batches = to_batches(test_set, 256, False)
    train_local, num_local, test_local = {}, {}, {}
    for cid in range(3):
        train_local[cid] = to_batches(Subset(train_set, shards[cid]), batch_size, True)
        num_local[cid]   = len(shards[cid])
        test_local[cid]  = test_batches
    train_global = [b for cid in range(3) for b in train_local[cid]]
    return [len(train_set), len(test_set), train_global, test_batches,
            num_local, train_local, test_local, 10]

fedml_acc = []
client_idx_log = []
class AccCollector(logging.Handler):
    def emit(self, record):
        try:
            msg = record.getMessage()
        except Exception:
            return
        if "'test_acc'" in msg:
            m = re.search(r"'test_acc':\s*([0-9.]+)", msg)
            if m: fedml_acc.append(float(m.group(1)))
        if "client_indexes = [" in msg:
            m2 = re.search(r"client_indexes = \[([^\]]+)\]", msg)
            if m2: client_idx_log.append(m2.group(1).strip())

def run_fedml_3clients(config_path):
    sys.argv = ["fedml_3clients.py", "--cf", config_path]
    args = fedml.init()                       # логирование FedML уже настроено
    device = fedml.device.get_device(args)
    dataset = build_3client_mnist(args.batch_size)
    print(">>> клиентов:", len(dataset[5]), "| примеров на клиента:", dataset[4])
    handler = AccCollector()
    logging.getLogger().addHandler(handler)   # вешаем ПОСЛЕ init
    logging.getLogger().setLevel(logging.INFO)
    try:
        FedMLRunner(args, device, dataset, MLP()).run()
    finally:
        logging.getLogger().removeHandler(handler)

fedml_acc.clear(); client_idx_log.clear()
run_fedml_3clients("fedml_config_3clients.yaml")
print("Прогон завершён")

[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:41.009104490] [INFO] [__init__.py:164:init] args.rank = 0, args.worker_num = 3
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:41.010714054] [INFO] [ml_engine_adapter.py:147:get_torch_device] args = <fedml.arguments.Arguments object at 0x0000019E0DAB0ED0>, using_gpu = False, device_id = 0, device_type = cpu
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:41.012484550] [INFO] [device.py:49:get_device] device = cpu
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:58.318388462] [INFO] [2390588997.py:54:run_fedml_3clients] >>> клиентов:
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:58.320378541] [INFO] [2390588997.py:54:run_fedml_3clients]  
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:58.320378541] [INFO] [2390588997.py:54:run_fedml_3clients] 3
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:58.321379423] [INFO] [2390588997.py:54:run_fedml_3clients]  
[FedML-Client @device-id-0] [Tue, 28 Jul 2026 06:15:58.3223

In [4]:
accs = [round(a * 100, 2) for a in fedml_acc]
print("client_indexes по раундам:", client_idx_log[:6])
print("FedML (3 фикс. клиента, IID) accuracy по раундам:", accs)

with open("fedml_3clients_accuracy.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["round", "test_acc_%"])
    for i, a in enumerate(accs):
        w.writerow([i, a])
print("Сохранено: fedml_3clients_accuracy.csv")

client_indexes по раундам: ['0, 1, 2', '0, 1, 2', '0, 1, 2', '0, 1, 2', '0, 1, 2', '0, 1, 2']
FedML (3 фикс. клиента, IID) accuracy по раундам: [92.66, 93.62, 94.26, 94.73, 95.2]
Сохранено: fedml_3clients_accuracy.csv
